# Web Scraping with Python

**Learning Objectives:**
- Understand how HTTP requests retrieve web pages and how HTML structures content
- Use the `requests` library to fetch web pages programmatically
- Parse HTML with `BeautifulSoup` to extract text, links, and attributes
- Scrape tabular data into pandas DataFrames (manually and via `pd.read_html()`)
- Handle pagination, errors, and ethical considerations in scraping workflows

---


## Using this notebook in Google Colab

1. Before editing, select **File > Save a copy in Drive**.
2. Run the cells in order. Some practice cells intentionally wait for your input.
3. The notebook creates any course folders and teaching files it needs automatically.
4. Files under `/content` are temporary and disappear when the Colab runtime resets.
5. Never paste an API key into a notebook cell. Use the **Secrets** panel when instructed.

You do not need to find, copy, or type a file path for the prepared course data.


## 1. What is Web Scraping?

Web scraping is the automated extraction of data from websites. Websites are designed for human consumption -- they present information as formatted HTML pages. Scraping converts that unstructured HTML into structured data that programs can work with, such as CSV files or pandas DataFrames.

**When to scrape:**
- No API is available for the data you need
- You need to collect data from multiple pages or sites
- The data is publicly visible but not downloadable

**When NOT to scrape:**
- An official API or data download exists (always prefer these)
- The site's Terms of Service prohibit it
- The data is personal or behind a login

The general pipeline is: **Inspect** the page → **Fetch** the HTML → **Parse** it → **Extract** the data → **Store** it.


## 2. How the Web Works (Quick Refresher)

When you visit a URL in your browser, the following happens:

1. Your browser sends an **HTTP GET request** to the server hosting the website.
2. The server responds with **HTML** (plus CSS, JavaScript, images, etc.).
3. The browser **renders** the HTML into the visual page you see.

When we scrape, our Python script plays the role of the browser -- it sends the GET request and receives the raw HTML. Instead of rendering it visually, we parse and extract data from the HTML tags.


### Key HTML Tags to Know

| Tag | Purpose | Example |
|-----|---------|---------|
| `<h1>` to `<h6>` | Headings | `<h1>Page Title</h1>` |
| `<p>` | Paragraph | `<p>Some text</p>` |
| `<a>` | Hyperlink | `<a href="url">Link</a>` |
| `<table>`, `<tr>`, `<td>` | Table, row, cell | Tabular data |
| `<div>` | Generic container | Groups content |
| `<span>` | Inline container | Wraps text |
| `<ul>`, `<li>` | Unordered list | List items |
| `<img>` | Image | `<img src="photo.jpg">` |

Every tag can have **attributes** like `class`, `id`, `href`, and `src`. These attributes are how we target specific elements when scraping.


## 3. Fetching Pages with `requests`

The `requests` library handles the HTTP communication. Let's start by installing the required libraries and fetching our first page.


In [ ]:
%pip install -q beautifulsoup4 lxml

import requests


class TeachingSession(requests.Session):
    """A requests session with a classroom-safe default timeout."""

    def request(self, method, url, **kwargs):
        kwargs.setdefault("timeout", 15)
        return super().request(method, url, **kwargs)


session = TeachingSession()
session.headers.update(
    {"User-Agent": "ISA383-AUS educational web-scraping notebook"}
)

print("Web-scraping packages are ready.")


In [2]:
# Install if needed (uncomment the line below)
# !pip install requests beautifulsoup4

import requests

In [ ]:
# Fetch a page
url = "https://quotes.toscrape.com/"
response = session.get(url)
response.raise_for_status()

# Check the status code
print("Status code:", response.status_code)
print("Content length:", len(response.text), "characters")


**Status codes to know:**
- `200` -- Success
- `404` -- Page not found
- `403` -- Forbidden (access denied)
- `500` -- Server error

Always check the status code before parsing. A non-200 response means you did not get the page you expected.


In [4]:
# Preview the raw HTML (first 500 characters)
print(response.text[:5000])

<!DOCTYPE html>
<html lang="en">
<head>
	<meta charset="UTF-8">
	<title>Quotes to Scrape</title>
    <link rel="stylesheet" href="/static/bootstrap.min.css">
    <link rel="stylesheet" href="/static/main.css">
    
    
</head>
<body>
    <div class="container">
        <div class="row header-box">
            <div class="col-md-8">
                <h1>
                    <a href="/" style="text-decoration: none">Quotes to Scrape</a>
                </h1>
            </div>
            <div class="col-md-4">
                <p>
                
                    <a href="/login">Login</a>
                
                </p>
            </div>
        </div>
    

<div class="row">
    <div class="col-md-8">

    <div class="quote" itemscope itemtype="http://schema.org/CreativeWork">
        <span class="text" itemprop="text">“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”</span>
        <span>by <small class="auth

### Try It Yourself 1: Fetch a Different Site

Now it's your turn. Fetch the homepage of books.toscrape.com and check the status code. Print the first 300 characters of HTML to get a sense of the structure.

**What to look for:** Notice the `<article>` tags and `<h3>` elements that contain book information.

In [ ]:
# Try It Yourself 1: Fetch books.toscrape.com
# TODO 1: Define the URL for books.toscrape.com
# url = ...

# TODO 2: Send a GET request and store the response
# response = ...

# TODO 3: Print the status code
# print("Status code:", ...)

# TODO 4: Print the first 300 characters of the HTML
# print(response.text[:300])

That raw HTML is what we need to parse. Notice the tags, attributes, and nested structure -- this is what BeautifulSoup will help us navigate.


## 4. Parsing HTML with BeautifulSoup

BeautifulSoup takes the raw HTML string and converts it into a navigable tree structure. You can then search for elements by tag name, class, id, or CSS selectors.


In [6]:
from bs4 import BeautifulSoup

# Create a soup object
soup = BeautifulSoup(response.text, "html.parser")

# Pretty-print a snippet
print(soup.prettify()[:400])

<!DOCTYPE html>
<html lang="en">
 <head>
  <meta charset="utf-8"/>
  <title>
   Quotes to Scrape
  </title>
  <link href="/static/bootstrap.min.css" rel="stylesheet"/>
  <link href="/static/main.css" rel="stylesheet"/>
 </head>
 <body>
  <div class="container">
   <div class="row header-box">
    <div class="col-md-8">
     <h1>
      <a href="/" style="text-decoration: none">
       Quotes to Scr


### `find()` and `find_all()`

- `find(tag, attrs)` returns the **first** matching element (or `None`)
- `find_all(tag, attrs)` returns a **list** of all matching elements


In [7]:
# Find the first <h1> tag
title = soup.find("h1")
print("Title:", title.text)

Title: 
Quotes to Scrape



In [8]:
# Find all <span> tags with class "text" (these hold the quotes)
quote_spans = soup.find_all("span", class_="text")

# Print the first 3 quotes
for span in quote_spans[:3]:
    print(span.text)
    print("---")

“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”
---
“It is our choices, Harry, that show what we truly are, far more than our abilities.”
---
“There are only two ways to live your life. One is as though nothing is a miracle. The other is as though everything is a miracle.”
---


In [9]:
quote_spans[0].text

'“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”'

### Extracting Attributes

Tags have attributes you can access like dictionary keys.


In [10]:
# Find the first <a> (link) tag on the page
link = soup.find("a")
print("Text:", link.text)
print("href:", link["href"])

Text: Quotes to Scrape
href: /


In [11]:
# Extract all links from the page
all_links = soup.find_all("a")
for a in all_links[:5]:
    print(a.get("href", "no href"), "--", a.text.strip())

/ -- Quotes to Scrape
/login -- Login
/author/Albert-Einstein -- (about)
/tag/change/page/1/ -- change
/tag/deep-thoughts/page/1/ -- deep-thoughts


### CSS Selectors with `select()`

The `select()` method uses CSS selector syntax, which can be more concise:

| Selector | Meaning |
|----------|---------|
| `tag` | All elements of that tag |
| `.class` | Elements with that class |
| `#id` | Element with that id |
| `parent child` | Descendant elements |


In [12]:
# Using CSS selectors
quotes_css = soup.select("span.text")
authors_css = soup.select("small.author")

for q, a in zip(quotes_css[:5], authors_css[:5]):
    print(f"{a.text}: {q.text}")

Albert Einstein: “The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”
J.K. Rowling: “It is our choices, Harry, that show what we truly are, far more than our abilities.”
Albert Einstein: “There are only two ways to live your life. One is as though nothing is a miracle. The other is as though everything is a miracle.”
Jane Austen: “The person, be it gentleman or lady, who has not pleasure in a good novel, must be intolerably stupid.”
Marilyn Monroe: “Imperfection is beauty, madness is genius and it's better to be absolutely ridiculous than absolutely boring.”


### Try It Yourself 2: Find Elements on a New Page

Practice using `find()` and `find_all()` on the books page. Each book on books.toscrape.com is inside an `<article class="product_pod">`. The title lives in an `<h3>` tag containing an `<a>` tag whose `title` attribute holds the full book name. The price is in a `<p class="price_color">` tag.

**What to look for:** Books are wrapped in `<article>` tags with the class `product_pod`.

In [ ]:
# Try It Yourself 2: Parse books.toscrape.com

# First, fetch and parse (this part is done for you)
url = "https://books.toscrape.com/"
response = session.get(url)
response.raise_for_status()
soup = BeautifulSoup(response.text, "html.parser")

# TODO 1: Find ALL article tags with class "product_pod"
# books = soup.find_all(...)
# print(f"Found {len(books)} books on the page")

# TODO 2: From the FIRST book, extract the title
# HINT: book.find("h3").find("a")["title"]
# first_book = books[0]
# title = ...
# print("First book:", title)

# TODO 3: From the FIRST book, extract the price text
# HINT: Look for a <p> with class "price_color"
# price = ...
# print("Price:", price)

# TODO 4: Use a CSS selector to get ALL prices on the page
# HINT: soup.select("p.price_color")
# all_prices = ...
# for p in all_prices[:5]:
#     print(p.text)


## 5. Practical Example: Scraping Quotes

Let's put it all together. We will scrape all quotes from the first page of [quotes.toscrape.com](https://quotes.toscrape.com/) and store them in a pandas DataFrame.

**Step 1: Inspect the page.** Each quote is inside a `<div class="quote">` that contains:
- `<span class="text">` -- the quote text
- `<small class="author">` -- the author name
- `<a class="tag">` -- tags associated with the quote


In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

# Fetch and parse
url = "https://quotes.toscrape.com/"
response = session.get(url)
response.raise_for_status()
soup = BeautifulSoup(response.text, "html.parser")

# Find all quote containers
quote_divs = soup.find_all("div", class_="quote")

# Extract data from each quote
data = []
for div in quote_divs:
    text = div.find("span", class_="text").text
    author = div.find("small", class_="author").text
    tags = [tag.text for tag in div.find_all("a", class_="tag")]
    data.append({
        "quote": text,
        "author": author,
        "tags": ", ".join(tags)
    })

# Convert to DataFrame
df = pd.DataFrame(data)
print(f"Scraped {len(df)} quotes from page 1")
df.head()


### Try It Yourself 3: Build a Book DataFrame

Now put the full pipeline together. Scrape all 20 books from the first page of books.toscrape.com into a DataFrame with columns: `title`, `price`, and `rating`. The star rating is encoded as a CSS class on a `<p>` tag inside each book, e.g. `<p class="star-rating Three">`. The second class name (One, Two, Three, Four, Five) is the rating.

**What to look for:** Each book has a `<p>` tag with class `star-rating` followed by another class representing the star count.

In [ ]:
# Try It Yourself 3: Full scraping pipeline on books.toscrape.com

import requests
from bs4 import BeautifulSoup

url = "https://books.toscrape.com/"
response = session.get(url)
response.raise_for_status()
soup = BeautifulSoup(response.text, "html.parser")
books = soup.find_all("article", class_="product_pod")
book_data = []

# TODO: Loop through books and extract title, price, and star rating.
# Hint: the rating is the second class on the p.star-rating tag.
# TODO: Append one dictionary per book, then create and display a DataFrame.


## 6. Scraping Tables

HTML tables are one of the most common targets for scraping. You can extract them manually with BeautifulSoup, or use the convenient `pd.read_html()` shortcut.


### Manual Table Extraction


In [ ]:
# Example: scraping a table from a practice page
url = "https://www.scrapethissite.com/pages/simple/"
response = session.get(url)
response.raise_for_status()
soup = BeautifulSoup(response.text, "html.parser")

# Find all country entries (each is a div with class "country")
countries = soup.find_all("div", class_="col-md-4 country")

country_data = []
for c in countries[:5]:  # first 5 for demonstration
    name = c.find("h3", class_="country-name").text.strip()
    capital = c.find("span", class_="country-capital").text.strip()
    population = c.find("span", class_="country-population").text.strip()
    country_data.append({
        "name": name,
        "capital": capital,
        "population": population
    })

pd.DataFrame(country_data)


### The Easy Way: `pd.read_html()`

Pandas can extract `<table>` elements directly from a URL or HTML string. It returns a list of DataFrames, one per table found on the page.


In [ ]:
# Let's try it on a Wikipedia page
from io import StringIO

url = "https://en.wikipedia.org/wiki/List_of_countries_by_GDP_(PPP)"
response = session.get(url, headers={"User-Agent": "Mozilla/5.0"})
response.raise_for_status()

print(response)


In [21]:
response.text[0:500]

'<!DOCTYPE html>\n<html class="client-nojs vector-feature-language-in-header-enabled vector-feature-language-in-main-menu-disabled vector-feature-language-in-main-page-header-disabled vector-feature-page-tools-pinned-disabled vector-feature-toc-pinned-clientpref-1 vector-feature-main-menu-pinned-disabled vector-feature-limited-width-clientpref-1 vector-feature-limited-width-content-enabled vector-feature-custom-font-size-clientpref-1 vector-feature-appearance-pinned-clientpref-1 skin-theme-clientp'

In [ ]:
StringIO(response.text)

'<!DOCTYPE html>\n<html class="client-nojs vector-feature-language-in-header-enabled vector-feature-language-in-main-menu-disabled vector-feature-language-in-main-page-header-disabled vector-feature-page-tools-pinned-disabled vector-feature-toc-pinned-clientpref-1 vector-feature-main-menu-pinned-disabled vector-feature-limited-width-clientpref-1 vector-feature-limited-width-content-enabled vector-feature-custom-font-size-clientpref-1 vector-feature-appearance-pinned-clientpref-1 skin-theme-clientpref-day vector-sticky-header-enabled vector-toc-available skin-theme-clientpref-thumb-standard" lang="en" dir="ltr">\n<head>\n<meta charset="UTF-8">\n<title>List of countries by GDP (PPP) - Wikipedia</title>\n<script>(function(){var className="client-js vector-feature-language-in-header-enabled vector-feature-language-in-main-menu-disabled vector-feature-language-in-main-page-header-disabled vector-feature-page-tools-pinned-disabled vector-feature-toc-pinned-clientpref-1 vector-feature-main-me

DDOS Attack: Distributed Denial of Service. 
- Overwhelming the server with too many requests

In [ ]:
# Extract every HTML table from the downloaded page.
# Select the largest table rather than assuming a fixed page position.
tables = pd.read_html(StringIO(response.text))
print(f"Found {len(tables)} tables on the page")

df_gdp = max(tables, key=lambda table: table.shape[0]).copy()
print("Selected table shape:", df_gdp.shape)
display(df_gdp.head())


### Try It Yourself 4: Extract a Wikipedia Table

Try `pd.read_html()` yourself. The page at `https://en.wikipedia.org/wiki/List_of_largest_companies_by_revenue` contains a table of the world's largest companies. Use `pd.read_html()` to extract it, then select the first table and display the first 10 rows.

**What to look for:** Wikipedia tables are standard `<table>` tags with `<tr>` rows and `<td>` cells.

In [ ]:
# Try It Yourself 4: Table extraction with pd.read_html()

# TODO 1: Use pd.read_html() to extract all tables from the Wikipedia URL above
# url = "https://en.wikipedia.org/wiki/List_of_largest_companies_by_revenue"
# tables = ...

# TODO 2: Print how many tables were found
# print(f"Found {len(tables)} tables")

# TODO 3: Select the most relevant table (usually index 0 or 1) and display the first 10 rows
# df = tables[...]
# df.head(10)

## 7. Handling Multiple Pages (Pagination)

Most websites split data across multiple pages. Two common strategies exist:
1. **Known number of pages** -- loop through page numbers
2. **Unknown pages** -- follow the "Next" link until it disappears


In [ ]:
# Strategy 1: Loop through known page numbers
import time

all_quotes = []

for page in range(1, 4):  # scrape pages 1, 2, and 3
    url = f"https://quotes.toscrape.com/page/{page}/"
    response = session.get(url)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")
    
    for div in soup.find_all("div", class_="quote"):
        text = div.find("span", class_="text").text
        author = div.find("small", class_="author").text
        all_quotes.append({"quote": text, "author": author})
    
    print(f"Page {page}: scraped {len(soup.find_all('div', class_='quote'))} quotes")
    time.sleep(1)  # be polite -- wait 1 second between requests

df_all = pd.DataFrame(all_quotes)
print(f"\nTotal: {len(df_all)} quotes from 3 pages")


In [36]:
df_all

,quote,author
0,“The world as we have created it is a process ...,Albert Einstein
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling
2,“There are only two ways to live your life. On...,Albert Einstein
3,"“The person, be it gentleman or lady, who has ...",Jane Austen
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe
5,“Try not to become a man of success. Rather be...,Albert Einstein
6,“It is better to be hated for what you are tha...,André Gide
7,"“I have not failed. I've just found 10,000 way...",Thomas A. Edison
8,“A woman is like a tea bag; you never know how...,Eleanor Roosevelt
9,"“A day without sunshine is like, you know, nig...",Steve Martin


In [ ]:
# Strategy 2: Follow the "Next" link
all_quotes_v2 = []
url = "https://quotes.toscrape.com/"

while url:
    response = session.get(url)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")
    
    for div in soup.find_all("div", class_="quote"):
        text = div.find("span", class_="text").text
        author = div.find("small", class_="author").text
        all_quotes_v2.append({"quote": text, "author": author})
    
    # Check for a "Next" button
    next_li = soup.find("li", class_="next")
    if next_li:
        next_href = next_li.find("a")["href"]
        url = "https://quotes.toscrape.com" + next_href
    else:
        url = None  # no more pages
    
    time.sleep(1)

print(f"Scraped {len(all_quotes_v2)} quotes across all pages")


In [38]:
pd.DataFrame(all_quotes_v2)

,quote,author
0,“The world as we have created it is a process ...,Albert Einstein
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling
2,“There are only two ways to live your life. On...,Albert Einstein
3,"“The person, be it gentleman or lady, who has ...",Jane Austen
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe
...,...,...
95,“You never really understand a person until yo...,Harper Lee
96,“You have to write the book that wants to be w...,Madeleine L'Engle
97,“Never tell the truth to people who are not wo...,Mark Twain
98,"“A person's a person, no matter how small.”",Dr. Seuss


### Try It Yourself 5: Paginate Through Books

Scrape books from the first 3 pages of books.toscrape.com. The URL pattern is `https://books.toscrape.com/catalogue/page-{n}.html` for pages 2 onward. Page 1 is simply `https://books.toscrape.com/`. Collect the title and price of every book across all 3 pages into a single DataFrame.

**What to look for:** Page 1 has a different URL structure than pages 2+. Check the href in pagination links to find the pattern.

In [ ]:
# Try It Yourself 5: Multi-page scraping on books.toscrape.com
import time

all_books = []

# TODO: Create the three page URLs.
# Hint: page 1 uses the site root; pages 2 and 3 use catalogue/page-{n}.html.
# TODO: Fetch each page, extract title and price, and append dictionaries to all_books.
# TODO: Pause for one second between requests.
# TODO: Create a DataFrame and report how many books you collected.

## 8. Error Handling

Real-world scraping encounters connection timeouts, missing pages, and unexpected HTML. Writing robust code means handling these gracefully.


In [ ]:
# Robust fetching with error handling
def fetch_page(url, timeout=10):
    """Fetch a URL and return a BeautifulSoup object, or None on failure."""
    try:
        response = session.get(url, timeout=timeout)
        response.raise_for_status()  # raises HTTPError for 4xx/5xx
        print(response)
        return BeautifulSoup(response.text, "html.parser")
    except requests.exceptions.RequestException as e:
        print(f"Failed to fetch {url}: {e}")
        return None

# Test with a valid page
soup = fetch_page("https://quotes.toscrape.com/")
if soup:
    print("Success! Found", len(soup.find_all("div", class_="quote")), "quotes")

# Test with a bad URL
soup_bad = fetch_page("https://quotes.toscrape.com/nonexistent")


In [41]:
soup_bad = fetch_page("https://quotes.toscrape.com/page/12/")

<Response [200]>


In [ ]:
# Safe extraction: always check for None before accessing .text
def safe_text(tag, default="N/A"):
    """Extract text from a tag, returning default if tag is None."""
    return tag.text.strip() if tag else default

# Example usage
div = soup.find("div", class_="quote")
print(safe_text(div.find("span", class_="text")))
print(safe_text(div.find("span", class_="nonexistent")))

“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”
N/A


### Try It Yourself 6: Add Error Handling to Your Scraper

Wrap your book scraper in proper error handling. Write a function `fetch_books(url)` that fetches a page, handles request failures with try/except, and safely extracts title and price (returning 'N/A' if a tag is missing). Test it on a valid URL and an invalid one.

**What to look for:** Use try/except to catch `requests.exceptions.RequestException` errors and always validate that tags exist before accessing their text.

In [ ]:
# Try It Yourself 6: Robust book scraper

def fetch_books(url):
    """Fetch a page of books and return title/price dictionaries."""
    # TODO: Catch requests.exceptions.RequestException.
    # TODO: Call raise_for_status() before parsing the response.
    # TODO: Check that tags exist before reading attributes or text.
    # TODO: Return an empty list after a failed request.
    pass

# TODO: Test the function with one valid URL and one invalid URL.

## 9. Ethics and `robots.txt`

Before scraping any website, check its `robots.txt` file. This file tells automated agents which parts of the site they are allowed to access.

**Rules of responsible scraping:**
1. Check `robots.txt` (at `example.com/robots.txt`)
2. Read the Terms of Service
3. Add a delay between requests (`time.sleep(1)`)
4. Do not scrape personal or sensitive data
5. Prefer APIs when available
6. Cache pages so you do not re-download


In [ ]:
# Check robots.txt for our practice site
robots = session.get("https://quotes.toscrape.com/robots.txt")
print(robots.text)


---

## 10. Comprehensive Exercises

Complete the following exercises to reinforce your understanding.

### Exercise 1: Extract All Authors
Scrape [quotes.toscrape.com](https://quotes.toscrape.com/) and create a list of **unique authors**. Print the authors sorted alphabetically. How many unique authors are on page 1?


In [ ]:
# Your solution for Exercise 1


### Exercise 2: Multi-Page Book Scraper
Scrape the first **3 pages** of [books.toscrape.com](https://books.toscrape.com/). For each book, extract the title, price, and star rating. Store everything in a DataFrame. The star rating is encoded as a CSS class on a `<p>` tag (e.g., `<p class="star-rating Three">`).

*Hint:* The URL pattern for pages is `https://books.toscrape.com/catalogue/page-{n}.html`.


In [ ]:
# Your solution for Exercise 2


### Exercise 3: Table Extraction Challenge
Use `pd.read_html()` to extract data from a Wikipedia page of your choice that contains a table. Clean the resulting DataFrame by:
1. Selecting only relevant columns
2. Renaming columns to something meaningful
3. Handling any missing values

Display the first 10 rows of your cleaned DataFrame.


In [ ]:
# Your solution for Exercise 3


### Exercise 4: Build a Reusable Scraper Function
Write a function `scrape_quotes(num_pages)` that:
1. Accepts the number of pages to scrape
2. Fetches each page with a 1-second delay
3. Handles errors gracefully (skip failed pages)
4. Returns a pandas DataFrame with columns: `quote`, `author`, `tags`

Test it by scraping 5 pages.


In [ ]:
# Your solution for Exercise 4


### Exercise 5: Competitive Price Comparison (Open-Ended)
Visit [books.toscrape.com](https://books.toscrape.com/) and scrape the **Travel** category. Extract all book titles and prices. Then answer:
1. What is the average price of travel books?
2. Which is the most expensive travel book?
3. What is the price range (min to max)?

*Hint:* Navigate to the Travel category page first and check its URL structure.


In [ ]:
# Your solution for Exercise 5


---

## Summary

In this notebook, we covered the full web scraping pipeline in Python:

- **`requests`** fetches raw HTML from a URL using HTTP GET requests.
- **`BeautifulSoup`** parses that HTML into a navigable tree that you can search with `find()`, `find_all()`, and `select()`.
- **Extracting data** means pulling `.text` content and attribute values (like `href` and `src`) from the tags you find.
- **`pd.read_html()`** is a convenient shortcut for extracting well-structured HTML tables directly into DataFrames.
- **Pagination** is handled by looping through page numbers or following "Next" links.
- **Error handling** (timeouts, missing elements, bad status codes) makes your scraper robust.
- **Ethics**: always check `robots.txt`, respect rate limits, and prefer official APIs when available.

**Key libraries:** `requests`, `beautifulsoup4` (imported as `bs4`), `pandas`
